# ML-KEM-1024 · PYNQ-Z2 AXI verification

Place this notebook, `mlkem1024.py`, `kat_vectors.json`, `release_manifest.json`, `k4_accel.bit`, and the paired `k4_accel.hwh` in one directory on the board. Open this notebook with the board's PYNQ Python 3.6+ kernel. No vendor source tree is required.

This notebook downloads the overlay and runs public official ACVP vectors. Saved notebook outputs are intentionally empty: no board result is claimed until these cells run on the board. `KATP` is a completion marker; the host must also verify exact output bytes, signed return codes, and hardware command/transfer counts. These regressions are not formal CAVP certification.

## 1. Verify the release and load the overlay
All manifest file hashes and the HWH address range are checked before download. The driver then checks the hardware ID/ABI/clock/RAM size and explicitly holds the CPU in reset. The AXI window is `0x40000000`, length `0x40000`; the processing clock is 100 MHz.

In [ ]:
from pathlib import Path
from mlkem1024 import MLKEM1024

BUNDLE = Path.cwd()
board = MLKEM1024.load(BUNDLE / 'release_manifest.json')
print('Overlay verified and CPU reset asserted.')
print('Manifest SHA-256:', board.artifacts['manifest_sha256'])
print('Prepared official cases:', len(board.vectors['cases']))

## 2. Run the representative official subset
Original indices `0`, `1`, and `115` cover KeyGen, Encaps, and Decaps. Every transaction resets the CPU, uploads one input-only fixture, starts execution, and verifies the result. A trap, timeout, firmware failure, output mismatch, or wrong hardware count produces `FAIL` and stops that run. Expected negative key-check return codes in the full suite are valid passing results.

In [ ]:
subset = board.run_cases([0, 1, 115], timeout=10.0)
print(subset['status'], subset['unique_checked_cases'], '/ 145 unique official cases')
for case in subset['cases']:
    print(case['case_index'], case['operation'], 'PASS' if case['passed'] else case['error'])

## 3. Optional complete 145-case run
Set `RUN_FULL_145 = True` explicitly to run every original index. Only one successful verification of each unique index `0..144` within this run earns `FULL_PASS`. The representative subset alone always remains `SUBSET_PASS`.

In [ ]:
RUN_FULL_145 = False
full = None
if RUN_FULL_145 and subset['passed']:
    full = board.run_cases(range(145), timeout=10.0)
    print(full['status'], full['unique_checked_cases'], '/ 145 unique official cases')
elif RUN_FULL_145:
    print('Full run skipped because the subset failed; export the failure evidence below.')
else:
    print('Full run is disabled. Set RUN_FULL_145 = True to enable it.')

## 4. API cycles and transport/wall time
**Live API time** is the firmware's raw `rdcycle` interval divided by 100 MHz. The empty bracket is also reported separately; API time excludes PS upload/download and host verification. **Transport** is measured upload plus download time; **run wait** includes CPU startup and host polling; **wall time** covers the whole host transaction. Operation 4 includes seed KeyGen followed by Decaps.

The comparison column is a **historical RTL CPU-only baseline** from the frozen `rv32im_fast` results. It is not a live CPU-only board measurement. No measured board speedup is claimed until both the CPU-only and accelerated overlays are measured on the board under matching conditions.

In [ ]:
from html import escape
from IPython.display import HTML, display

report = full if full is not None else subset
columns = ['Index', 'Operation', 'Live API cycles', 'Live API ms', 'API minus empty ms',
           'Transport ms', 'Run wait ms', 'Host wall ms', 'Historical RTL CPU-only cycles']
rows = []
for case in report['cases']:
    if not case['passed']:
        print('FAILED:', case['case_index'], case['error_type'], case['error'])
        continue
    rows.append([case['case_index'], case['operation'], case['raw_cycles'],
                 f"{case['algorithm_seconds'] * 1000:.3f}",
                 f"{case['algorithm_seconds_minus_empty'] * 1000:.3f}",
                 f"{(case['upload_seconds'] + case['download_seconds']) * 1000:.3f}",
                 f"{case['run_wait_seconds'] * 1000:.3f}",
                 f"{case['wall_seconds'] * 1000:.3f}",
                 case['historical_rtl_cpu_only']['raw_cycles']])
header = '<tr>' + ''.join('<th>' + escape(label) + '</th>' for label in columns) + '</tr>'
body = ''.join('<tr>' + ''.join('<td>' + escape(str(value)) + '</td>' for value in row) + '</tr>'
               for row in rows)
display(HTML('<table>' + header + body + '</table>'))
print('Report status:', report['status'], '| Total host wall seconds:', report['total_wall_seconds'])

## 5. Export reproducible evidence
Export both successful and failed attempts. Each JSON includes the bit/HWH/vector hashes, release provenance, UTC timestamps, runtime details, original case identities, actual output bytes, signed returns, cycle counters, hardware counts, timing definitions, and coverage. Existing evidence is never overwritten.

In [ ]:
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
for label, saved_report in [('subset', subset), ('full', full)]:
    if saved_report is not None:
        path = board.export(saved_report, BUNDLE / 'evidence' / f'{stamp}_{label}.json')
        print(path, saved_report['status'])

Developer preparation (Python 3.10+, run in the repository on the developer PC):
```text
python pynq/mlkem1024/mlkem1024.py --prepare --output <bundle-directory>
python pynq/mlkem1024/mlkem1024.py --self-test
```
Preparation validates the pinned public ACVP sources and freezes the 145 inputs, host-only expected outputs, hardware call expectations, and historical RTL cycles into `kat_vectors.json`. The release exporter creates the manifest after pairing the implemented `.bit` and `.hwh`. The fake-MMIO tests do not program hardware.